# Using DSPy framework

Using anthropic model

In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd
import re

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")

Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


In [2]:
from langchain_community.graphs import Neo4jGraph

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)

enhanced_schema = enhanced_graph.schema

print(enhanced_schema)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"


Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `modelPhenotypeLabel`: STRING 
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentarget

In [3]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

In [4]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [20]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        start = time.time()
        result = run_with_timeout(query_cypher_graph, 60, graph, llm_output)
        duration = time.time() - start
        return {
            "query":llm_output,
            "success":True,
            "result": list(result),
            "time": duration
        }
    except Exception as e:
        return {
            "query":llm_output,
            "result": f"Failed to execute query: {str(e)}",
            "success":False,
            "exception":str(e)
        }


In order to use DSPy prompt optimization capabilities we need a good evaluation dataset, so we'll use biomix for test and evaluation.

As a base query we will use an enchanced LLM schema that is provided by the LangChain.

In [6]:
# DSPy setup:

import dspy

llm = dspy.LM("anthropic/claude-3-5-sonnet-20240620", max_tokens = 2000)
dspy.settings.configure(lm = llm)

d:\AppData\conda_envs\pistoia\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading test-set

In [7]:
import random

questions_test = pd.read_csv("biomix_true_false_selected_augmented.csv").rename(columns={"text":"question" , "label": "answer"}).sample(frac=1, random_state=42).reset_index(drop=True)
questions_train = pd.read_csv("biomix_true_false_selected_augmented_2.csv").rename(columns={"text":"question" , "label": "answer"}).sample(frac=1, random_state=42).reset_index(drop=True)


testset = [dspy.Example(statement=x['question'], answer=x['answer']).with_inputs("statement") for _, x in questions_test.iterrows()]
trainset = [dspy.Example(statement=x['question'], answer=x['answer']).with_inputs("statement") for _, x in questions_train.iterrows()]


## Evaluating DSPy strategies

### 9.1 Naive QA

Just predicting truthfulness of a statement

In [8]:
# First version is a simple QA to answer question:

predict = dspy.Predict("statement -> answer")
answer = predict(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    answer='This statement is incorrect. Polycythemia Vera is strongly associated with mutations in the JAK2 gene.\n\nPolycythemia Vera (PV) is a myeloproliferative neoplasm characterized by increased red blood cell production. The JAK2 gene plays a crucial role in the development of this condition:\n\n1. JAK2 V617F mutation: Approximately 95% of patients with PV have a specific mutation in the JAK2 gene called V617F.\n\n2. JAK2 exon 12 mutations: In the remaining 5% of PV cases without the V617F mutation, most have mutations in exon 12 of the JAK2 gene.\n\nThese JAK2 mutations lead to constitutive activation of the JAK-STAT signaling pathway, resulting in uncontrolled cell proliferation and the clinical features of Polycythemia Vera.\n\nThe strong association between JAK2 mutations and Polycythemia Vera is well-established in medical literature and is a key aspect of the diagnosis and understanding of this disease.'
)

In [11]:
# Doing the same but with signatures:

from pydantic import BaseModel, Field

class AnswerCorrectness(BaseModel):
    correctness: bool = Field(description="Correctness of the statement")
    explanation: str = Field(description="Explanation of the correctness")
    

class QACorrectness(dspy.Signature):
    """Given the statement, evaluate its correctness"""
    statement: str = dspy.InputField(desc="The statement to be evaluated")
    answer: AnswerCorrectness = dspy.OutputField()

predict = dspy.Predict(QACorrectness)
answer = predict(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    answer=AnswerCorrectness(correctness=False, explanation='This statement is incorrect. Polycythemia Vera is actually strongly associated with mutations in the JAK2 gene. The JAK2 V617F mutation is found in approximately 95% of patients with Polycythemia Vera. This mutation leads to constitutive activation of the JAK-STAT signaling pathway, resulting in increased production of red blood cells, which is a hallmark of Polycythemia Vera. The association between JAK2 mutations and Polycythemia Vera is so well-established that JAK2 testing is a key diagnostic criterion for this myeloproliferative neoplasm.')
)

In [14]:
# Evaluating with dspy Evaluate
from dspy.evaluate import Evaluate
from dspy.evaluate.metrics import answer_exact_match

def validate_answer(example, pred, trace=None):
    return example.answer == pred.answer.correctness

evaluate_program = Evaluate(devset = testset, metric=validate_answer, display_progress=True, display_table=10, provide_traceback=True)

In [22]:
eval = evaluate_program(predict)
print(eval)

Average Metric: 4.00 / 100 (4.0%):   4%|▍         | 4/100 [00:00<00:00, 215.35it/s]

Average Metric: 95.00 / 100 (95.0%): 100%|██████████| 100/100 [00:00<00:00, 356.87it/s]

2024/12/15 13:10:28 INFO dspy.evaluate.evaluate: Average Metric: 95 / 100 (95.0%)


,statement,example_answer,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,correctness=False explanation='This statement is incorrect. Argini...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,correctness=False explanation='This statement is incorrect. Cherub...,
2,Cystinuria is not associated with Gene GP1BB,True,correctness=True explanation='The statement is correct. Cystinuria...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,correctness=False explanation='This statement is incorrect. Pierso...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,correctness=False explanation='This statement is incorrect. Mastoc...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,correctness=True explanation='The statement is correct. Argininosu...,✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,correctness=True explanation='The statement is correct. Bernard-So...,✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,correctness=False explanation='This statement is incorrect. CHARGE...,✔️ [True]
8,Progeria associates Gene LMNA,True,"correctness=True explanation=""The statement 'Progeria associates G...",✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,correctness=False explanation='This statement is incorrect. Polycy...,✔️ [True]


95.0


### 9.2 Naive Chain Of Thought

Think step-by-step; output reflection

In [23]:
# chain of thought

cot = dspy.ChainOfThought(QACorrectness)
answer = cot(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    reasoning='To evaluate the correctness of this statement, we need to consider the current medical understanding of Polycythemia Vera (PV) and its genetic associations.\n\nPolycythemia Vera is a myeloproliferative neoplasm characterized by increased production of red blood cells. It is well-established in medical literature that PV is strongly associated with mutations in the JAK2 gene.\n\nThe JAK2 gene provides instructions for making a protein that is part of a signaling pathway called the JAK/STAT pathway, which is important for controlling blood cell production. Mutations in the JAK2 gene, particularly the V617F mutation, are found in approximately 95% of patients with Polycythemia Vera.\n\nThis strong association between PV and JAK2 mutations is a key diagnostic criterion for the disease. In fact, the World Health Organization (WHO) includes the presence of a JAK2 mutation as one of the major criteria for diagnosing Polycythemia Vera.\n\nGiven this information, the 

In [24]:
eval = evaluate_program(cot)
print(eval)

Average Metric: 98.00 / 100 (98.0%): 100%|██████████| 100/100 [11:00<00:00,  6.61s/it]

2024/12/15 13:21:37 INFO dspy.evaluate.evaluate: Average Metric: 98 / 100 (98.0%)


,statement,example_answer,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"To evaluate the correctness of this statement, we need to consider...",correctness=False explanation='The statement is incorrect. Arginin...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"To evaluate the correctness of this statement, we need to consider...",correctness=True explanation='The statement is correct. Cherubism ...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,"To evaluate the correctness of this statement, we need to consider...",correctness=True explanation='The statement is correct. Cystinuria...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,"To evaluate the correctness of this statement, we need to consider...",correctness=False explanation='The statement is incorrect. Pierson...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,"To evaluate the correctness of this statement, we need to consider...",correctness=False explanation='The statement is incorrect. Mastocy...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,"To evaluate the correctness of this statement, let's break it down...",correctness=True explanation='The statement is correct. Argininosu...,✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"To evaluate the correctness of this statement, let's break it down...",correctness=True explanation='The statement is correct. Bernard-So...,✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,"To evaluate the correctness of this statement, we need to consider...",correctness=False explanation='The statement is incorrect. CHARGE ...,✔️ [True]
8,Progeria associates Gene LMNA,True,"To evaluate the correctness of this statement, let's consider the ...",correctness=True explanation='The statement is correct. Progeria i...,✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,"To evaluate the correctness of this statement, we need to consider...",correctness=False explanation='The statement is incorrect. Polycyt...,✔️ [True]


98.0


In [25]:
dspy.inspect_history(1)





[2024-12-15T13:21:37.744973]

System message:

Your input fields are:
1. `statement` (str): The statement to be evaluated

Your output fields are:
1. `reasoning` (str)
2. `answer` (AnswerCorrectness)

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object", "properties": {"correctness": {"type": "boolean", "description": "Correctness of the statement", "title": "Correctness"}, "explanation": {"type": "string", "description": "Explanation of the correctness", "title": "Explanation"}}, "required": ["correctness", "explanation"], "title": "AnswerCorrectness"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Given the statement, evaluate its correctness


User message:

[[ ## statement ## ]]
Adenine pho

### 9.3 Optimized Chain of Thought

Using MIPROv2 optimizer to prompt-optimize CoT

In [26]:
tp = dspy.MIPROv2(metric = validate_answer, auto="light")
optimized_cot = tp.compile(cot, trainset=trainset)

2024/12/15 13:21:37 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 5
valset size: 79

2024/12/15 13:21:46 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/15 13:21:46 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/15 13:21:46 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=5 sets of demonstrations...


Bootstrapping set 1/5
Bootstrapping set 2/5
Bootstrapping set 3/5


 20%|██        | 4/20 [00:24<01:39,  6.19s/it]


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/5


 15%|█▌        | 3/20 [00:16<01:34,  5.59s/it]


...

2024/12/15 13:30:33 INFO dspy.evaluate.evaluate: Average Metric: 25 / 25 (100.0%)
2024/12/15 13:30:33 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 1'].
2024/12/15 13:30:33 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
2024/12/15 13:30:33 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [98.73]
2024/12/15 13:30:33 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 98.73
2024/12/15 13:30:33 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/15 13:30:33 INFO dspy.teleprompt.mipro_optimizer_v2: == Minibatch Trial 7 / 7 ==



Average Metric: 25.00 / 25 (100.0%): 100%|██████████| 25/25 [02:11<00:00,  5.27s/it]

2024/12/15 13:32:45 INFO dspy.evaluate.evaluate: Average Metric: 25 / 25 (100.0%)
2024/12/15 13:32:45 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 4', 'Predictor 0: Few-Shot Set 4'].
2024/12/15 13:32:45 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
2024/12/15 13:32:45 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [98.73]
2024/12/15 13:32:45 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 98.73
2024/12/15 13:32:45 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/15 13:32:45 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Full Eval 1 =====
2024/12/15 13:32:45 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 100.0) from minibatch trials...



Average Metric: 79.00 / 79 (100.0%): 100%|██████████| 79/79 [01:40<00:00,  1.27s/it]

2024/12/15 13:34:25 INFO dspy.evaluate.evaluate: Average Metric: 79 / 79 (100.0%)
2024/12/15 13:34:25 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 100.0
2024/12/15 13:34:25 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [98.73, 100.0]
2024/12/15 13:34:25 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 100.0
2024/12/15 13:34:25 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2024/12/15 13:34:25 INFO dspy.teleprompt.mipro_optimizer_v2: 

2024/12/15 13:34:25 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 100.0!


In [27]:
eval_optimized = evaluate_program(optimized_cot)
print(eval_optimized)

Average Metric: 100.00 / 100 (100.0%): 100%|██████████| 100/100 [13:33<00:00,  8.13s/it]

2024/12/15 13:47:58 INFO dspy.evaluate.evaluate: Average Metric: 100 / 100 (100.0%)


,statement,example_answer,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"To evaluate this statement, let's break it down and analyze the co...",correctness=False explanation='Argininosuccinic Aciduria is not as...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"To evaluate this statement, let's break it down step-by-step: 1. T...",correctness=True explanation='The statement is correct. Cherubism ...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,"To evaluate this statement, let's break it down step-by-step: 1. T...",correctness=True explanation='The statement is correct. Cystinuria...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,"To evaluate this statement, let's break it down and analyze it ste...",correctness=False explanation='The statement is incorrect. Pierson...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,"To evaluate this statement, let's break it down and analyze it ste...",correctness=False explanation='The statement is incorrect. Mastocy...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,"To evaluate this statement, let's break it down and analyze it ste...",correctness=True explanation='The statement is correct. Argininosu...,✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"To evaluate this statement, let's break it down and analyze it ste...",correctness=True explanation='The statement is correct. Bernard-So...,✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,"To evaluate this statement, let's break it down and analyze the ke...",correctness=False explanation='CHARGE Syndrome is not associated w...,✔️ [True]
8,Progeria associates Gene LMNA,True,"To evaluate this statement, let's break it down and analyze it ste...","correctness=True explanation='The statement is correct. Progeria, ...",✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,"To evaluate this statement, let's consider the known facts about P...",correctness=False explanation='The statement is incorrect. Polycyt...,✔️ [True]


100.0


In [28]:
optimized_cot.save("09.3-optimized_cot-claude.json", save_field_meta=True)

In [29]:
dspy.inspect_history(n=1)





[2024-12-15T13:47:58.770943]

System message:

Your input fields are:
1. `statement` (str): The statement to be evaluated

Your output fields are:
1. `reasoning` (str)
2. `answer` (AnswerCorrectness)

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object", "properties": {"correctness": {"type": "boolean", "description": "Correctness of the statement", "title": "Correctness"}, "explanation": {"type": "string", "description": "Explanation of the correctness", "title": "Explanation"}}, "required": ["correctness", "explanation"], "title": "AnswerCorrectness"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        You are a genetic disease expert tasked with evaluating the correctness of statements about gene

### 9.4 QA with cypher retrieval

This will work like RAG, but instead of semantic retrieval we will use Cypher retrieval

In [21]:
class GenerateCypher(dspy.Signature):
    """Generate cypher statement to check correctness of a statement"""
    statement = dspy.InputField()
    schema = dspy.InputField(desc="Schema of a neo4j database")
    cypher = dspy.OutputField(desc="Valid cypher query that can be used to check correctness of the statement")

cot_cypher = dspy.ChainOfThought(GenerateCypher)
answer = cot_cypher(statement = "Polycythemia Vera is not associated with Gene JAK2", schema=enhanced_schema)
answer



d:\AppData\conda_envs\pistoia\Lib\site-packages\pydantic\_internal\_fields.py:172: UserWarning: Field name "schema" in "GenerateCypher" shadows an attribute in parent "Signature"
  warnings.warn(
d:\AppData\conda_envs\pistoia\Lib\site-packages\pydantic\_internal\_fields.py:172: UserWarning: Field name "schema" in "StringSignature" shadows an attribute in parent "Signature"
  warnings.warn(


Prediction(
    reasoning="To check if Polycythemia Vera is not associated with Gene JAK2, we need to search for any existing relationships between these two entities in the database. If we find any associations, it would contradict the statement. We'll use the Gene and Disease nodes, looking for connections through GeneToDiseaseAssociation relationships.",
    cypher='MATCH (g:Gene {approvedSymbol: "JAK2"})\nMATCH (d:Disease {name: "Polycythemia Vera"})\nMATCH (g)-[:IS_PART_OF]->(assoc:GeneToDiseaseAssociation)<-[:IS_PART_OF]-(d)\nRETURN g.approvedSymbol AS Gene, d.name AS Disease, COUNT(assoc) AS AssociationCount'
)

In [22]:
# More complicated module similar to RAG

class QACorrectnessContext(dspy.Signature):
    """Given the statement and cypher results, evaluate correctness of the statement with respect to the data in the graph. Do not use prior knowledge of biology"""
    statement: str = dspy.InputField(desc="The statement to be evaluated")
    context = dspy.InputField(desc="Results of cypher query to check the validity of statemet agains ground truth")
    answer: AnswerCorrectness = dspy.OutputField()

class GraphRAG(dspy.Module):

    def __init__(self, schema):
        super().__init__()
        self.generate_query  = dspy.ChainOfThought(GenerateCypher)
        self.generate_answer = dspy.ChainOfThought(QACorrectnessContext)
        self.schema = schema
    
    def forward(self, statement):
        query = self.generate_query(statement=statement, schema=self.schema)
        result = query_graph(query.cypher)
        query_result = str(result['result'])
        answer = self.generate_answer(statement=statement, context=query_result)
        return answer
    

graph_rag = GraphRAG(enhanced_schema)
answer = graph_rag(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer



Prediction(
    reasoning='Based on the provided context, which is an empty list, there is no information available in the graph database about the relationship between Polycythemia Vera and the JAK2 gene. Without any data to support or refute the statement, we cannot determine its correctness. The lack of information in the graph does not necessarily mean that there is no association between Polycythemia Vera and JAK2; it only indicates that this particular database does not contain any relevant data about this relationship.',
    answer=AnswerCorrectness(correctness=False, explanation='There is insufficient information in the provided context to evaluate the statement. The empty result from the cypher query suggests that the graph database does not contain any data about the relationship between Polycythemia Vera and the JAK2 gene. Therefore, we cannot determine whether the statement is correct or incorrect based solely on this lack of information.')
)

In [24]:
evaluate_program(graph_rag)

Average Metric: 73.00 / 100 (73.0%): 100%|██████████| 100/100 [08:19<00:00,  4.99s/it]

2024/12/16 16:16:30 INFO dspy.evaluate.evaluate: Average Metric: 73 / 100 (73.0%)


,statement,example_answer,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"I apologize, but I cannot evaluate the correctness of the statemen...",correctness=False explanation='Unable to verify the statement due ...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"Based on the provided context from the cypher query results, the s...",correctness=True explanation='The cypher query results confirm tha...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,"Based on the provided context, we can see that there are multiple ...",correctness=True explanation='The statement is correct according t...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,"Based on the provided context, the statement ""Pierson syndrome is ...","correctness=False explanation=""The statement is incorrect. The con...",✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,"Based on the provided context, which states ""The statement is corr...",correctness=False explanation='The statement is incorrect. Accordi...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,"I apologize, but I cannot evaluate the correctness of the statemen...",correctness=False explanation='Unable to determine the correctness...,
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"Based on the provided context, we can see that there is a record s...",correctness=True explanation='The statement is correct. The contex...,✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,"Based on the provided context, we cannot determine the correctness...",correctness=False explanation='Unable to verify the statement due ...,✔️ [True]
8,Progeria associates Gene LMNA,True,"I apologize, but I cannot evaluate the correctness of the statemen...",correctness=False explanation='Unable to determine the correctness...,
9,Polycythemia Vera is not associated with Gene JAK2,False,"Based on the provided context, which is an empty list, there is no...",correctness=False explanation='There is insufficient information i...,✔️ [True]


73.0

In [25]:
answer = graph_rag(statement = "Argininosuccinic Aciduria associates Gene ASL	")
answer


Prediction(
    reasoning="I apologize, but I cannot evaluate the correctness of the statement based on the given context. The context provided indicates that there was an error in executing the Cypher query, and no actual data about diseases or genes was returned. Without any relevant information from the graph database, it's impossible to verify whether Argininosuccinic Aciduria associates with the Gene ASL or not.\n\nTo properly evaluate this statement, we would need successful query results that include information about Argininosuccinic Aciduria and its associated genes, or information about the ASL gene and its associated diseases. Since we don't have this information, we cannot make a determination about the correctness of the statement.",
    answer=AnswerCorrectness(correctness=False, explanation='Unable to determine the correctness of the statement due to a failed database query. The provided context does not contain any relevant information about Argininosuccinic Aciduria or

In [26]:
dspy.inspect_history(2)





[2024-12-16T16:17:21.061180]

System message:

Your input fields are:
1. `statement` (str)
2. `schema` (str): Schema of a neo4j database

Your output fields are:
1. `reasoning` (str)
2. `cypher` (str): Valid cypher query that can be used to check correctness of the statement

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## schema ## ]]
{schema}

[[ ## reasoning ## ]]
{reasoning}

[[ ## cypher ## ]]
{cypher}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Generate cypher statement to check correctness of a statement


User message:

[[ ## statement ## ]]
Argininosuccinic Aciduria associates Gene ASL	

[[ ## schema ## ]]
Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `v

errors are probably due to incorrect cypher statements

### 9.5. Optimized QC with cypher retrieval

In [27]:
tp = dspy.MIPROv2(metric = validate_answer, auto="light")
optimized_rag = tp.compile(graph_rag, trainset=trainset)

2024/12/16 16:18:38 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 3
valset size: 79

2024/12/16 16:18:40 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/16 16:18:40 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/16 16:18:40 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


 25%|██▌       | 5/20 [01:24<04:13, 16.88s/it]
2024/12/16 16:20:04 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2024/12/16 16:20:04 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.
2024/12/16 16:20:05 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing instructions...



Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.


2024/12/16 16:20:50 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2024/12/16 16:20:50 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Generate cypher statement to check correctness of a statement

2024/12/16 16:20:50 INFO dspy.teleprompt.mipro_optimizer_v2: 1: You are a lead researcher at a prestigious genetic research institute, tasked with validating critical statements about gene-disease associations. Lives depend on the accuracy of your work. Given a statement about a genetic disease and its association with a specific gene, along with the schema of our comprehensive Neo4j database, your mission is to generate a precise Cypher query.

This query must rigorously check the correctness of the statement against our vast genetic data repository. Remember, your query will be the foundation for life-altering medical decisions and groundbreaking research. Inaccuracy could lead to misdiagnoses or misdirected studies, potentially harming patients and setting ba

...

2024/12/16 16:39:22 ERROR dspy.utils.parallelizer: Error processing item Example({'statement': 'Hyperargininemia is associated with Gene LMX1B', 'answer': False}) (input_keys={'statement'}): litellm.RateLimitError: AnthropicException - {"type":"error","error":{"type":"rate_limit_error","message":"This request would exceed your organization’s rate limit of 160,000 input tokens per minute. For details, refer to: https://docs.anthropic.com/en/api/rate-limits; see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase."}}. Set `provide_traceback=True` to see the stack trace.


Average Metric: 47.00 / 53 (88.7%):  68%|██████▊   | 54/79 [02:45<00:52,  2.10s/it]

2024/12/16 16:40:54 ERROR dspy.utils.parallelizer: Error processing item Example({'statement': 'Gray Platelet Syndrome is not associated with Gene NBEAL2', 'answer': False}) (input_keys={'statement'}): litellm.RateLimitError: AnthropicException - {"type":"error","error":{"type":"rate_limit_error","message":"This request would exceed your organization’s rate limit of 160,000 input tokens per minute. For details, refer to: https://docs.anthropic.com/en/api/rate-limits; see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase."}}. Set `provide_traceback=True` to see the stack trace.


Average Metric: 71.00 / 77 (92.2%): 100%|██████████| 79/79 [03:30<00:00,  2.66s/it]

2024/12/16 16:41:38 INFO dspy.evaluate.evaluate: Average Metric: 71.0 / 79 (89.9%)
2024/12/16 16:41:38 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 89.87
2024/12/16 16:41:38 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [74.68, 89.87]
2024/12/16 16:41:38 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 89.87
2024/12/16 16:41:38 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2024/12/16 16:41:38 INFO dspy.teleprompt.mipro_optimizer_v2: 

2024/12/16 16:41:38 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 89.87!


In [28]:
evaluate_program(optimized_rag)
optimized_rag.save("09.5-optimized_rag_claude.json", save_field_meta=True)

Average Metric: 91.00 / 100 (91.0%): 100%|██████████| 100/100 [24:02<00:00, 14.43s/it]

2024/12/16 17:15:01 INFO dspy.evaluate.evaluate: Average Metric: 91 / 100 (91.0%)


,statement,example_answer,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"Based on the provided context, we can see that there is a record l...",correctness=False explanation='While there is a record linking Arg...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"Based on the provided context, we can see that there is a record f...",correctness=True explanation='The statement is correct according t...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,"To evaluate the statement ""Cystinuria is not associated with Gene ...",correctness=True explanation='The provided context does not show a...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,The statement claims that Pierson syndrome is not associated with ...,correctness=False explanation='The statement is incorrect because ...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,"Based on the provided context, we can see that there are multiple ...","correctness=True explanation=""The context provides multiple record...",
5,Argininosuccinic Aciduria associates Gene ASL,True,"Based on the provided context, we can see that there are multiple ...","correctness=True explanation=""The context provides multiple record...",✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"Based on the provided context, we can see that there are multiple ...","correctness=True explanation=""The context provides multiple record...",✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,"Based on the provided context, there is no evidence of an associat...",correctness=False explanation='The context does not provide any ev...,✔️ [True]
8,Progeria associates Gene LMNA,True,"Based on the provided context, we can see that there is a strong a...",correctness=True explanation='The context provides multiple record...,✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,"Based on the provided context, we can see that there are multiple ...",correctness=False explanation='The statement is incorrect. The con...,✔️ [True]


### 9.6 QA with ReAct

In [29]:
class AnswerCorrectness(BaseModel):
    correctness: bool = Field(description="Correctness of the statement")
    explanation: str = Field(description="Explanation of the correctness")
    

class QACorrectness(dspy.Signature):
    """Given the statement, evaluate its correctness using information in the knowledge graph"""
    statement: str = dspy.InputField(desc="The statement to be evaluated")
    answer: AnswerCorrectness = dspy.OutputField()

def run_cypher_query(cypher_statement:str) -> str:
    """Given the cypher statement, run the query and return the result"""
    result = query_graph(cypher_statement)
    return result

naive_react = dspy.ReAct(QACorrectness, tools = [run_cypher_query])
answer = naive_react(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    trajectory={'thought_0': 'To evaluate the correctness of the statement "Polycythemia Vera is not associated with Gene JAK2", we need to query the knowledge graph for any relationships between Polycythemia Vera and the JAK2 gene. Let\'s start by searching for these entities and their potential associations.', 'tool_name_0': 'run_cypher_query', 'tool_args_0': {'cypher_statement': "MATCH (d:Disease {name: 'Polycythemia Vera'})-[r]-(g:Gene {symbol: 'JAK2'}) RETURN d, r, g"}, 'observation_0': {'query': "MATCH (d:Disease {name: 'Polycythemia Vera'})-[r]-(g:Gene {symbol: 'JAK2'}) RETURN d, r, g", 'success': True, 'result': [], 'time': 0.1708672046661377}, 'thought_1': "The query we ran didn't return any results, which might suggest that there's no direct relationship between Polycythemia Vera and the JAK2 gene in our knowledge graph. However, before we conclude that the statement is correct, we should perform a more comprehensive search. Let's try to find any indirect connecti

In [ ]:
dspy.inspect_history(2)





[2024-12-16T17:15:45.989135]

System message:

Your input fields are:
1. `statement` (str): The statement to be evaluated
2. `trajectory` (str)

Your output fields are:
1. `next_thought` (str)
2. `next_tool_name` (Literal[run_cypher_query, finish])
3. `next_tool_args` (dict[str, Any])

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## trajectory ## ]]
{trajectory}

[[ ## next_thought ## ]]
{next_thought}

[[ ## next_tool_name ## ]]
{next_tool_name}        # note: the value you produce must be one of: run_cypher_query; finish

[[ ## next_tool_args ## ]]
{next_tool_args}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Given the statement, evaluate its correctness using information in the knowledge graph
        
        You will be given `statement`

In [31]:
evaluate_program(naive_react)

Average Metric: 65.00 / 100 (65.0%): 100%|██████████| 100/100 [47:09<00:00, 28.30s/it]

2024/12/16 18:18:18 INFO dspy.evaluate.evaluate: Average Metric: 65 / 100 (65.0%)


,statement,example_answer,trajectory,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"{'thought_0': 'To evaluate the correctness of the statement ""Argin...","Based on our queries to the knowledge graph, we found that: 1. The...","correctness=False explanation=""The statement cannot be verified us...",✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"{'thought_0': 'To evaluate the correctness of the statement ""Cheru...","Based on the queries performed on our knowledge graph, we found th...",correctness=False explanation='There is insufficient information i...,
2,Cystinuria is not associated with Gene GP1BB,True,"{'thought_0': 'To evaluate the correctness of the statement ""Cysti...","Based on the queries performed on the knowledge graph, we can conc...",correctness=True explanation='The knowledge graph contains no evid...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,"{'thought_0': 'To evaluate the correctness of the statement ""Piers...","Based on the queries performed on the knowledge graph, we can conc...",correctness=False explanation='The statement cannot be evaluated a...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,"{'thought_0': 'To evaluate the correctness of the statement ""Masto...","Based on the queries performed on the knowledge graph, we can draw...","correctness=True explanation=""Based on the information available i...",
5,Argininosuccinic Aciduria associates Gene ASL,True,"{'thought_0': 'To evaluate the correctness of the statement ""Argin...","Based on the queries performed on the knowledge graph, we were una...","correctness=False explanation=""The statement cannot be verified us...",
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"{'thought_0': 'To evaluate the correctness of the statement ""Berna...","Based on the queries performed on the knowledge graph, we were una...","correctness=False explanation=""Based on the available information ...",
7,CHARGE Syndrome is associated with Gene APRT,False,"{'thought_0': 'To evaluate the correctness of the statement ""CHARG...","Based on the queries performed on the knowledge graph, we can conc...",correctness=False explanation='The statement is incorrect accordin...,✔️ [True]
8,Progeria associates Gene LMNA,True,"{'thought_0': 'To evaluate the correctness of the statement ""Proge...","After conducting multiple queries on the knowledge graph, we were ...","correctness=False explanation=""Based on the information available ...",
9,Polycythemia Vera is not associated with Gene JAK2,False,"{'thought_0': 'To evaluate the correctness of the statement ""Polyc...","Based on our queries to the knowledge graph, we have found that: 1...",correctness=False explanation='The statement cannot be evaluated a...,✔️ [True]


65.0

In [33]:
answer = naive_react(statement = "Polycythemia Vera is not associated with Gene JAK2")
dspy.inspect_history(2)





[2024-12-16T18:59:54.555437]

System message:

Your input fields are:
1. `statement` (str): The statement to be evaluated
2. `trajectory` (str)

Your output fields are:
1. `next_thought` (str)
2. `next_tool_name` (Literal[run_cypher_query, finish])
3. `next_tool_args` (dict[str, Any])

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## trajectory ## ]]
{trajectory}

[[ ## next_thought ## ]]
{next_thought}

[[ ## next_tool_name ## ]]
{next_tool_name}        # note: the value you produce must be one of: run_cypher_query; finish

[[ ## next_tool_args ## ]]
{next_tool_args}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Given the statement, evaluate its correctness using information in the knowledge graph
        
        You will be given `statement`

Doesn't really work as planned; need to do some optimizations

### 9.7 Optimized QA with ReAct

In [34]:
class AnswerCorrectness(BaseModel):
    correctness: bool = Field(description="Correctness of the statement")
    explanation: str = Field(description="Explanation of the correctness")
    

class QACorrectness(dspy.Signature):
    """Given the statement, evaluate its correctness using information in the knowledge graph"""
    statement: str = dspy.InputField(desc="The statement to be evaluated")
    answer: AnswerCorrectness = dspy.OutputField()

def run_cypher_query(cypher_statement:str) -> str:
    """Given the cypher statement, run the query and return the result"""
    result = query_graph(cypher_statement)
    return result

naive_react = dspy.ReAct(QACorrectness, tools = [run_cypher_query])
answer = naive_react(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    trajectory={'thought_0': 'To evaluate the correctness of the statement "Polycythemia Vera is not associated with Gene JAK2", we need to query the knowledge graph for any relationships between Polycythemia Vera and the JAK2 gene. Let\'s start by searching for these entities and their potential associations.', 'tool_name_0': 'run_cypher_query', 'tool_args_0': {'cypher_statement': "MATCH (d:Disease {name: 'Polycythemia Vera'})-[r]-(g:Gene {symbol: 'JAK2'}) RETURN d, r, g"}, 'observation_0': {'query': "MATCH (d:Disease {name: 'Polycythemia Vera'})-[r]-(g:Gene {symbol: 'JAK2'}) RETURN d, r, g", 'success': True, 'result': [], 'time': 0.11953210830688477}, 'thought_1': 'The query returned an empty result, which might suggest that there is no direct relationship between Polycythemia Vera and the JAK2 gene in our knowledge graph. However, before concluding, we should perform a more general query to ensure that both entities exist in our graph and to check for any indirect relati

In [35]:
tp = dspy.MIPROv2(metric = validate_answer, auto="light")
optimized_react = tp.compile(naive_react, trainset=trainset)

2024/12/16 19:02:27 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 3
valset size: 79

2024/12/16 19:02:30 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/16 19:02:30 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/16 19:02:30 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


 30%|███       | 6/20 [02:45<06:25, 27.54s/it]
2024/12/16 19:05:15 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2024/12/16 19:05:15 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.
2024/12/16 19:05:15 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing instructions...



Bootstrapped 4 full traces after 6 examples for up to 1 rounds, amounting to 6 attempts.


2024/12/16 19:07:06 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2024/12/16 19:07:06 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Given the statement, evaluate its correctness using information in the knowledge graph

You will be given `statement` and your goal is to finish with `answer`.

To do this, you will interleave Thought, Tool Name, and Tool Args, and receive a resulting Observation.

Thought can reason about the current situation, and Tool Name can be the following types:

(1) run_cypher_query, whose description is <desc>Given the cypher statement, run the query and return the result</desc>. It takes arguments {'cypher_statement': 'str'} in JSON format.
(2) finish, whose description is <desc>Signals that the final outputs, i.e. `answer`, are now available and marks the task as complete.</desc>. It takes arguments {} in JSON format.

2024/12/16 19:07:06 INFO dspy.teleprompt.mipro_optimizer_v2: 1: You are a leading geneticist working on a groun

...

2024/12/16 19:51:31 ERROR dspy.utils.parallelizer: Error processing item Example({'statement': 'Refsum Disease associates Gene PHYH', 'answer': True}) (input_keys={'statement'}): litellm.RateLimitError: AnthropicException - {"type":"error","error":{"type":"rate_limit_error","message":"This request would exceed your organization’s rate limit of 160,000 input tokens per minute. For details, refer to: https://docs.anthropic.com/en/api/rate-limits; see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You may also contact sales at https://www.anthropic.com/contact-sales to discuss your options for a rate limit increase."}}. Set `provide_traceback=True` to see the stack trace.


Average Metric: 6.00 / 17 (35.3%): 100%|██████████| 25/25 [06:21<00:00, 15.24s/it]

2024/12/16 19:53:41 INFO dspy.evaluate.evaluate: Average Metric: 6.0 / 25 (24.0%)
2024/12/16 19:53:41 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 24.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 2'].
2024/12/16 19:53:41 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [0.0, 60.0, 0.0, 0.0, 0.0, 76.0, 24.0]
2024/12/16 19:53:41 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [63.29]
2024/12/16 19:53:41 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 63.29
2024/12/16 19:53:41 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/16 19:53:41 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Full Eval 1 =====
2024/12/16 19:53:41 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 76.0) from minibatch trials...



Average Metric: 63.00 / 79 (79.7%): 100%|██████████| 79/79 [05:35<00:00,  4.24s/it]

2024/12/16 19:59:16 INFO dspy.evaluate.evaluate: Average Metric: 63 / 79 (79.7%)
2024/12/16 19:59:17 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 79.75
2024/12/16 19:59:17 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [63.29, 79.75]
2024/12/16 19:59:17 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 79.75
2024/12/16 19:59:17 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2024/12/16 19:59:17 INFO dspy.teleprompt.mipro_optimizer_v2: 

2024/12/16 19:59:17 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 79.75!


In [36]:
evaluate_program(optimized_react)

Average Metric: 50.00 / 100 (50.0%):  60%|██████    | 60/100 [33:40<18:49, 28.23s/it]

2024/12/16 20:44:52 ERROR dspy.utils.parallelizer: Error processing item Example({'statement': 'Coffin-Lowry syndrome is not associated with Gene CFTR', 'answer': True}) (input_keys={'statement'}): 1 validation error for dict[str,any]
  Input should be a valid dictionary [type=dict_type, input_value='MATCH (n) RETURN n LIMIT 1', input_type=str]
    For further information visit https://errors.pydantic.dev/2.9/v/dict_type
Stack trace:
Traceback (most recent call last):
  File "d:\AppData\conda_envs\pistoia\Lib\site-packages\dspy\utils\parallelizer.py", line 47, in wrapped
    return function(item)
           ^^^^^^^^^^^^^^
  File "d:\AppData\conda_envs\pistoia\Lib\site-packages\dspy\evaluate\evaluate.py", line 101, in process_item
    prediction = program(**example.inputs())
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AppData\conda_envs\pistoia\Lib\site-packages\dspy\utils\callback.py", line 202, in wrapper
    return fn(instance, *args, **kwargs)
           ^^^^^^^^^^^^^^^^

Average Metric: 81.00 / 100 (81.0%): 100%|██████████| 100/100 [56:39<00:00, 34.00s/it]

2024/12/16 21:07:31 INFO dspy.evaluate.evaluate: Average Metric: 81.0 / 100 (81.0%)


,statement,example_answer,trajectory,reasoning,pred_answer,validate_answer,answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"{'thought_0': 'To evaluate the statement ""Argininosuccinic Aciduri...",I apologize for the repeated failures in querying the knowledge gr...,"correctness=False explanation='Based on general medical knowledge,...",✔️ [True],NaN
1,Cherubism is not associated with Gene UBR1,True,"{'thought_0': 'To evaluate the statement ""Cherubism is not associa...",I apologize for the persistent errors in attempting to query the k...,"correctness=False explanation=""Due to technical difficulties in qu...",,NaN
2,Cystinuria is not associated with Gene GP1BB,True,"{'thought_0': 'To evaluate the statement ""Cystinuria is not associ...",Based on the trajectory of our attempts to query the knowledge gra...,"correctness=True explanation=""While we couldn't definitively confi...",✔️ [True],NaN
3,Pierson syndrome is not associated with Gene LAMB2,False,"{'thought_0': 'To evaluate the statement ""Pierson syndrome is not ...",I apologize for the persistent errors in attempting to query the k...,correctness=False explanation='Despite being unable to verify the ...,✔️ [True],NaN
4,Mastocytosis is not associated with Gene KIT,False,"{'thought_0': 'To evaluate the statement ""Mastocytosis is not asso...","Based on the attempts to query the knowledge graph, we encountered...","correctness=False explanation=""While we couldn't directly verify t...",✔️ [True],NaN
5,Argininosuccinic Aciduria associates Gene ASL,True,"{'thought_0': 'To evaluate the statement ""Argininosuccinic Aciduri...","I apologize, but I was unable to properly query the knowledge grap...","correctness=True explanation='Based on general medical knowledge, ...",✔️ [True],NaN
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"{'thought_0': 'To evaluate the statement ""Bernard-Soulier Syndrome...",Based on the error messages received from multiple attempts to que...,"correctness=True explanation=""Although I couldn't verify this info...",✔️ [True],NaN
7,CHARGE Syndrome is associated with Gene APRT,False,"{'thought_0': 'To evaluate the statement ""CHARGE Syndrome is assoc...",Based on the repeated failures to execute Cypher queries using var...,correctness=False explanation='While I cannot confirm this with th...,✔️ [True],NaN
8,Progeria associates Gene LMNA,True,"{'thought_0': 'To evaluate the statement ""Progeria associates Gene...","Based on the Cypher query results, we can conclude that neither ""P...","correctness=False explanation=""Based on the available information ...",,NaN
9,Polycythemia Vera is not associated with Gene JAK2,False,"{'thought_0': 'To evaluate the statement ""Polycythemia Vera is not...","Based on the Cypher query result, we can analyze the statement ""Po...","correctness=False explanation=""The statement is incorrect. While o...",✔️ [True],NaN


81.0

In [ ]:
dspy.inspect_history(2)





[2024-12-16T21:07:31.799689]

System message:

Your input fields are:
1. `statement` (str): The statement to be evaluated
2. `trajectory` (str)

Your output fields are:
1. `reasoning` (str)
2. `answer` (AnswerCorrectness)

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object", "properties": {"correctness": {"type": "boolean", "description": "Correctness of the statement", "title": "Correctness"}, "explanation": {"type": "string", "description": "Explanation of the correctness", "title": "Explanation"}}, "required": ["correctness", "explanation"], "title": "AnswerCorrectness"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Given the statement, evaluate its cor

In [ ]:
optimized_react.save("09.7-optimized_react_claude.json", save_field_meta=True)
optimized_react_evals = evaluate_program(optimized_react)

In [ ]:
dspy.inspect_history(2)